# PARC2026 予選配布リポジトリ セットアップ（Colab版）

公式配布リポジトリ [matsuolab/PARC2026_pre](https://github.com/matsuolab/PARC2026_pre) を
Colab上でセットアップし、Track1の疎通確認（ランダムポリシーでの評価実行）まで行う。

**方針(2026-08-04)**: まずColab無料枠で試す。SmolVLA(約4.5億パラメータ)は
公式が「Colab T4で完走」を保証している軽量モデルのため、Pi0.5のようなOOM問題は起きにくい想定。
詰まった場合のみRunPodへ切り替える（`docs/env_setup.md`にPi0.5時代の経緯を記録済み）。

**ランタイム > ランタイムのタイプを変更 > GPU** に設定してから実行すること
（`setup.sh`自体はCPU torchで足りる仕様だが、後続のSmolVLA推論確認でGPUを使うため）。

## 0. GPU確認

In [ ]:
!nvidia-smi

## 1. Python 3.10 の確保

`setup.sh` は `python3.10` を明示的に要求する。Colabのデフォルトはバージョンが異なる場合があるため、
存在確認し、なければ deadsnakes PPA から導入する。

In [ ]:
import subprocess

has_py310 = subprocess.run(['which', 'python3.10'], capture_output=True).returncode == 0
has_venv_module = subprocess.run(
    ['python3.10', '-c', 'import venv'], capture_output=True
).returncode == 0 if has_py310 else False

print('python3.10 found:', has_py310)
print('python3.10 venv module usable:', has_venv_module)

if not has_py310 or not has_venv_module:
    print('Installing python3.10 (+venv/distutils) via apt / deadsnakes PPA...')

In [ ]:
%%bash
# 常に実行する(python3.10本体だけあってpython3.10-venvが欠けているケースがあるため、
# "python3.10があるかどうか"だけで分岐すると venv 未導入を見逃す)
export DEBIAN_FRONTEND=noninteractive
export NEEDRESTART_MODE=a
apt-get update -qq
apt-get install -y -qq software-properties-common
add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
apt-get update -qq
apt-get install -y -qq python3.10 python3.10-venv python3.10-distutils

python3.10 --version
python3.10 -m venv --help > /dev/null && echo "python3.10 venv module OK"

## 2. 配布リポジトリをclone

In [ ]:
import os

REPO_DIR = '/content/PARC2026_pre'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/matsuolab/PARC2026_pre.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

## 3. `setup.sh` 実行（初回のみ、アセット取得含め10〜20分程度）

venv作成・依存インストール(CPU torch)・LIBERO-plus/LIBEROの取得とパッチ・アセットダウンロード・
`~/.libero/config.yaml` 生成を一括で行う。

In [ ]:
%%bash
cd /content/PARC2026_pre
# venv作成に失敗した形跡があれば作り直す(setup.shは既存venvディレクトリがあれば再作成しない仕様のため)
if [ -d venv ] && [ ! -f venv/bin/activate ]; then
  echo "壊れたvenvディレクトリを検出、削除して再作成します"
  rm -rf venv
fi
bash setup.sh

## 4. 疎通確認（ランダムポリシーで評価パイプラインを回す）

ここまでは学習不要。配布されたテンプレートのまま、Track1のexampleタスクで
評価パイプラインが最後まで動くことを確認する。

ポリシーサーバーをバックグラウンドで起動し、`pipeline` から接続する。

In [ ]:
%%bash --bg
cd /content/PARC2026_pre
source env.sh
python submission_template/policy_server.py --port 8000 > /content/policy_server.log 2>&1

In [ ]:
import time
time.sleep(5)
!curl -s http://localhost:8000/health

In [ ]:
%%bash
cd /content/PARC2026_pre
source env.sh
python -m pipeline --server-url http://localhost:8000 --track track1 --n-episodes 2 --max-steps 600

## 5. ログ確認（エラーが出た場合）

In [ ]:
!cat /content/policy_server.log

## 次のステップ

ここまで成功したら、`examples/smolvla_libero_spatial_lora.ipynb` を別のColabノートブックとして開き、
SmolVLAのLoRA追加学習を進める（`docs/strategy.md` ステップ2）。

詰まった場合は、エラーメッセージを控えて `docs/env_setup.md` の「つまずきポイント一覧」に
追記し、それでも解決しなければRunPodへの切り替えを検討する。